In [ ]:
# https://www.youtube.com/watch?v=aAy-B6KPld8&t=361s

In [ ]:
import sqlalchemy as sa
from sqlalchemy.orm import Mapped, mapped_column, sessionmaker, declarative_base

In [ ]:
db = sa.create_engine("sqlite:///:memory:")
Session = sessionmaker(bind=db)
Base = declarative_base()

In [ ]:
class User(Base):
    __tablename__ = "users"

    id: Mapped[int] = mapped_column(primary_key=True)
    username: Mapped[str]
    email: Mapped[str]

    def __repr__(self) -> str:
        return f"<User(id={self.id}, username={self.username}, email={self.email})>"


In [ ]:
Base.metadata.create_all(db)
user = User(username="Arjan", email="Arjan@arjancodes.com")

with Session() as session:
    session.add(user)
    session.commit()
    print(session.query(User).all())

In [ ]:
# https://docs.sqlalchemy.org/en/20/orm/quickstart.html

In [1]:
from typing import List
from typing import Optional
from sqlalchemy import ForeignKey
from sqlalchemy import String
from sqlalchemy.orm import DeclarativeBase
from sqlalchemy.orm import Mapped
from sqlalchemy.orm import mapped_column
from sqlalchemy.orm import relationship

- Define tables classes

In [2]:
class Base(DeclarativeBase):
    pass

In [3]:
class User(Base):
    __tablename__ = 'user_account'
    
    id : Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(30))
    fullname: Mapped[Optional[str]]
    addresses: Mapped[List["Address"]] = relationship(back_populates="user",cascade="all, delete-orphan")
    
    def __repr__(self) -> str:
        return f"User(id={self.id!r}, name={self.name!r}, fullname={self.fullname!r})"
    
class Address(Base):
    __tablename__ = "address"

    id: Mapped[int] = mapped_column(primary_key=True)
    email_address: Mapped[str]
    user_id: Mapped[int] = mapped_column(ForeignKey("user_account.id"))

    user: Mapped["User"] = relationship(back_populates="addresses")

    def __repr__(self) -> str:
        return f"Address(id={self.id!r}, email_address={self.email_address!r})"

- Create an Engine

In [4]:
from sqlalchemy import create_engine

In [5]:
engine = create_engine("sqlite://", echo=True)

- Create the tables

In [6]:
Base.metadata.create_all(engine)

2024-09-05 11:29:33,633 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2024-09-05 11:29:33,634 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("user_account")
2024-09-05 11:29:33,635 INFO sqlalchemy.engine.Engine [raw sql] ()
2024-09-05 11:29:33,636 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("user_account")
2024-09-05 11:29:33,636 INFO sqlalchemy.engine.Engine [raw sql] ()
2024-09-05 11:29:33,637 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("address")
2024-09-05 11:29:33,637 INFO sqlalchemy.engine.Engine [raw sql] ()
2024-09-05 11:29:33,638 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("address")
2024-09-05 11:29:33,638 INFO sqlalchemy.engine.Engine [raw sql] ()
2024-09-05 11:29:33,641 INFO sqlalchemy.engine.Engine 
CREATE TABLE user_account (
	id INTEGER NOT NULL, 
	name VARCHAR(30) NOT NULL, 
	fullname VARCHAR, 
	PRIMARY KEY (id)
)


2024-09-05 11:29:33,641 INFO sqlalchemy.engine.Engine [no key 0.00076s] ()
2024-09-05 11:29:33,644 INFO sqlalchemy.engine.

- Create Objects and Persist

In [7]:
from sqlalchemy.orm import Session

In [8]:
with Session(engine) as session:
    spongebob = User(
        name="spongebob",
        fullname="Spongebob Squarepants",
        addresses=[Address(email_address="spongebob@sqlalchemy.org")],
    )
    sandy = User(
        name="sandy",
        fullname="Sandy Cheeks",
        addresses=[
            Address(email_address="sandy@sqlalchemy.org"),
            Address(email_address="sandy@squirrelpower.org"),
        ],
    )
    patrick = User(name="patrick", fullname="Patrick Star")
    
    session.add_all([spongebob, sandy, patrick])
    session.commit()

2024-09-05 11:29:42,503 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2024-09-05 11:29:42,508 INFO sqlalchemy.engine.Engine INSERT INTO user_account (name, fullname) VALUES (?, ?) RETURNING id
2024-09-05 11:29:42,509 INFO sqlalchemy.engine.Engine [generated in 0.00016s (insertmanyvalues) 1/3 (ordered; batch not supported)] ('spongebob', 'Spongebob Squarepants')
2024-09-05 11:29:42,510 INFO sqlalchemy.engine.Engine INSERT INTO user_account (name, fullname) VALUES (?, ?) RETURNING id
2024-09-05 11:29:42,511 INFO sqlalchemy.engine.Engine [insertmanyvalues 2/3 (ordered; batch not supported)] ('sandy', 'Sandy Cheeks')
2024-09-05 11:29:42,512 INFO sqlalchemy.engine.Engine INSERT INTO user_account (name, fullname) VALUES (?, ?) RETURNING id
2024-09-05 11:29:42,512 INFO sqlalchemy.engine.Engine [insertmanyvalues 3/3 (ordered; batch not supported)] ('patrick', 'Patrick Star')
2024-09-05 11:29:42,514 INFO sqlalchemy.engine.Engine INSERT INTO address (email_address, user_id) VALUES (?, ?) RETURN

- Simple SELECT

In [9]:
from sqlalchemy import select

- Open a session

In [10]:
session = Session(engine)

In [11]:
stmt = select(User).where(User.name.in_(["spongebob","sandy"]))

for user in session.scalars(stmt):
    print(user)
    
session.scalars(stmt).all()

2024-09-05 11:29:56,874 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2024-09-05 11:29:56,876 INFO sqlalchemy.engine.Engine SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account 
WHERE user_account.name IN (?, ?)
2024-09-05 11:29:56,876 INFO sqlalchemy.engine.Engine [generated in 0.00057s] ('spongebob', 'sandy')
User(id=1, name='spongebob', fullname='Spongebob Squarepants')
User(id=2, name='sandy', fullname='Sandy Cheeks')
2024-09-05 11:29:56,879 INFO sqlalchemy.engine.Engine SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account 
WHERE user_account.name IN (?, ?)
2024-09-05 11:29:56,881 INFO sqlalchemy.engine.Engine [cached since 0.004712s ago] ('spongebob', 'sandy')


[User(id=1, name='spongebob', fullname='Spongebob Squarepants'),
 User(id=2, name='sandy', fullname='Sandy Cheeks')]

- SELECT with JOIN

In [12]:
stmt = (
    select(Address)
    .join(Address.user)
    .where(User.name == "sandy")
    .where(Address.email_address == "sandy@sqlalchemy.org")
)

sandy_address = session.scalars(stmt).one()

sandy_address

2024-09-05 11:30:15,473 INFO sqlalchemy.engine.Engine SELECT address.id, address.email_address, address.user_id 
FROM address JOIN user_account ON user_account.id = address.user_id 
WHERE user_account.name = ? AND address.email_address = ?
2024-09-05 11:30:15,474 INFO sqlalchemy.engine.Engine [generated in 0.00125s] ('sandy', 'sandy@sqlalchemy.org')


Address(id=2, email_address='sandy@sqlalchemy.org')

- Make Changes

In [13]:
stmt = select(User).where(User.name== "patrick")

In [14]:
patrick=session.scalar(stmt)
patrick

2024-09-05 11:30:30,006 INFO sqlalchemy.engine.Engine SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account 
WHERE user_account.name = ?
2024-09-05 11:30:30,008 INFO sqlalchemy.engine.Engine [generated in 0.00173s] ('patrick',)


User(id=3, name='patrick', fullname='Patrick Star')

In [15]:
patrick= session.scalars(stmt).one()
patrick

2024-09-05 11:30:32,144 INFO sqlalchemy.engine.Engine SELECT user_account.id, user_account.name, user_account.fullname 
FROM user_account 
WHERE user_account.name = ?
2024-09-05 11:30:32,145 INFO sqlalchemy.engine.Engine [cached since 2.139s ago] ('patrick',)


User(id=3, name='patrick', fullname='Patrick Star')

In [16]:
patrick.addresses.append(Address(email_address="patrickstar@sqlalchemy.org"))

2024-09-05 11:30:55,173 INFO sqlalchemy.engine.Engine SELECT address.id AS address_id, address.email_address AS address_email_address, address.user_id AS address_user_id 
FROM address 
WHERE ? = address.user_id
2024-09-05 11:30:55,175 INFO sqlalchemy.engine.Engine [generated in 0.00137s] (3,)


In [17]:
sandy_address.email_address ="sandy_cheeks@sqlalchemy.org"

In [18]:
session.commit()

2024-09-05 11:31:43,616 INFO sqlalchemy.engine.Engine UPDATE address SET email_address=? WHERE address.id = ?
2024-09-05 11:31:43,617 INFO sqlalchemy.engine.Engine [generated in 0.00104s] ('sandy_cheeks@sqlalchemy.org', 2)
2024-09-05 11:31:43,618 INFO sqlalchemy.engine.Engine INSERT INTO address (email_address, user_id) VALUES (?, ?)
2024-09-05 11:31:43,619 INFO sqlalchemy.engine.Engine [generated in 0.00051s] ('patrickstar@sqlalchemy.org', 3)
2024-09-05 11:31:43,620 INFO sqlalchemy.engine.Engine COMMIT


- Some Deletes

In [19]:
sandy = session.get(User,2)

2024-09-05 11:32:40,210 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2024-09-05 11:32:40,212 INFO sqlalchemy.engine.Engine SELECT user_account.id AS user_account_id, user_account.name AS user_account_name, user_account.fullname AS user_account_fullname 
FROM user_account 
WHERE user_account.id = ?
2024-09-05 11:32:40,213 INFO sqlalchemy.engine.Engine [generated in 0.00047s] (2,)


In [20]:
sandy.addresses.remove(sandy_address)

2024-09-05 11:32:49,648 INFO sqlalchemy.engine.Engine SELECT address.id AS address_id, address.email_address AS address_email_address, address.user_id AS address_user_id 
FROM address 
WHERE ? = address.user_id
2024-09-05 11:32:49,649 INFO sqlalchemy.engine.Engine [cached since 114.5s ago] (2,)


In [21]:
session.flush()

2024-09-05 11:32:52,128 INFO sqlalchemy.engine.Engine DELETE FROM address WHERE address.id = ?
2024-09-05 11:32:52,129 INFO sqlalchemy.engine.Engine [generated in 0.00125s] (2,)


In [22]:
session.delete(patrick)

2024-09-05 11:32:54,615 INFO sqlalchemy.engine.Engine SELECT user_account.id AS user_account_id, user_account.name AS user_account_name, user_account.fullname AS user_account_fullname 
FROM user_account 
WHERE user_account.id = ?
2024-09-05 11:32:54,616 INFO sqlalchemy.engine.Engine [cached since 14.4s ago] (3,)
2024-09-05 11:32:54,617 INFO sqlalchemy.engine.Engine SELECT address.id AS address_id, address.email_address AS address_email_address, address.user_id AS address_user_id 
FROM address 
WHERE ? = address.user_id
2024-09-05 11:32:54,618 INFO sqlalchemy.engine.Engine [cached since 119.4s ago] (3,)


In [23]:
session.commit()

2024-09-05 11:32:56,674 INFO sqlalchemy.engine.Engine DELETE FROM address WHERE address.id = ?
2024-09-05 11:32:56,675 INFO sqlalchemy.engine.Engine [cached since 4.547s ago] (4,)
2024-09-05 11:32:56,676 INFO sqlalchemy.engine.Engine DELETE FROM user_account WHERE user_account.id = ?
2024-09-05 11:32:56,676 INFO sqlalchemy.engine.Engine [generated in 0.00063s] (3,)
2024-09-05 11:32:56,677 INFO sqlalchemy.engine.Engine COMMIT
